# IT3091 — Machine Learning Project · Group 2026-AI-08K
## Member 4: Model 03 — LightGBM Regressor Pipeline
**Student Name:** Raashidh M.R.A.  
**Student ID:** IT24104169  
**Role:** Member 4  
**Assigned Work:** Model 03 LightGBM + Comparative Evaluation Lead

### CELL 1: Download Project from GitHub & Install LightGBM

In [ ]:
# ========================================================
# CELL 1: DOWNLOAD PROJECT FROM GITHUB & INSTALL LIGHTGBM
# ========================================================
import os

# 1. Download the repository from GitHub
if not os.path.exists("IT3091-House-Price-Prediction"):
    !git clone https://github.com/Atheek-Fareez/IT3091-House-Price-Prediction.git

# 2. Open the project folder
%cd IT3091-House-Price-Prediction

# 3. Pull the newest updates from your teammates
!git pull origin main

# 4. Install LightGBM and other machine learning tools
!pip install -q lightgbm scikit-learn pandas numpy matplotlib seaborn joblib

print("✅ Success! Your GitHub repository is downloaded and ready to use.")

### CELL 2: Import Production Libraries & Setup Paths

In [ ]:
# ========================================================
# CELL 2: IMPORT PRODUCTION LIBRARIES & SETUP PATHS
# ========================================================
import sys
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyRegressor
from sklearn.model_selection import KFold, cross_val_predict, cross_validate, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from lightgbm import LGBMRegressor
import joblib

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/member_03/preprocessing.py").is_file()), None)

if ROOT and str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.member_03.preprocessing import load_ames_data, build_preprocessor

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("✅ Repository root found at:", ROOT)
print("✅ All Python tools and Member 3 preprocessor imported successfully!")

### CELL 3: Load Data & Set Up 5-Fold Cross-Validation

In [ ]:
# ========================================================
# CELL 3: LOAD DATA & SET UP 5-FOLD CROSS-VALIDATION
# ========================================================
train_df, test_df = load_ames_data(".")

print(f"📊 Training Data: {train_df.shape[0]} houses, {train_df.shape[1]} columns")
print(f"📊 Test Data:     {test_df.shape[0]} houses, {test_df.shape[1]} columns")

X = train_df.drop(columns=["Id", "SalePrice"]).copy()
y = train_df["SalePrice"].astype(float).copy()
y_log = np.log1p(y)

RANDOM_STATE = 42
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("✅ Data is loaded. Target price is converted to log scale. 5 folds ready.")

### CELL 4: The Academic Baseline Benchmark

In [ ]:
# ========================================================
# CELL 4: THE ACADEMIC BASELINE (BENCHMARK)
# ========================================================
baseline_model = DummyRegressor(strategy="mean")
baseline_log_preds = cross_val_predict(baseline_model, X, y_log, cv=cv)
baseline_dollar_preds = np.expm1(baseline_log_preds)

baseline_rmse = float(np.sqrt(mean_squared_error(y, baseline_dollar_preds)))
baseline_mae = float(mean_absolute_error(y, baseline_dollar_preds))
baseline_r2 = float(r2_score(y, baseline_dollar_preds))

print("========================================")
print("🏛️ ACADEMIC BASELINE BENCHMARK RESULTS")
print("========================================")
print(f"Baseline RMSE: ${baseline_rmse:,.2f}")
print(f"Baseline MAE:  ${baseline_mae:,.2f}")
print(f"Baseline R²:   {baseline_r2:.4f}")
print("========================================")

### CELL 5: Build LightGBM Pipeline & Tune Hyperparameters

In [ ]:
# ========================================================
# CELL 5: BUILD LIGHTGBM PIPELINE & TUNE HYPERPARAMETERS
# ========================================================
preprocessor = build_preprocessor(configuration="unscaled", log_numeric=False, sparse_output=False)

lgbm_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LGBMRegressor(
        objective="regression",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    ))
])

param_grid = {
    "regressor__n_estimators": [200, 300],
    "regressor__learning_rate": [0.03, 0.05],
    "regressor__num_leaves": [20, 31, 40],
    "regressor__max_depth": [4, 6, -1],
    "regressor__subsample": [0.8, 1.0],
    "regressor__colsample_bytree": [0.7, 0.8]
}

print("🔄 Tuning LightGBM with 5-Fold Cross-Validation...")
grid_search = GridSearchCV(
    estimator=lgbm_pipe,
    param_grid=param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X, y_log)
best_lgbm_pipeline = grid_search.best_estimator_

print("\n🏆 BEST HYPERPARAMETERS FOUND:")
for k, v in grid_search.best_params_.items():
    print(f"  • {k}: {v}")

### CELL 6: Evaluate Tuned Model & Overfitting Check

In [ ]:
# ========================================================
# CELL 6: EVALUATE TUNED MODEL & OVERFITTING CHECK
# ========================================================
lgbm_log_preds = cross_val_predict(best_lgbm_pipeline, X, y_log, cv=cv)
lgbm_dollar_preds = np.expm1(lgbm_log_preds)

lgbm_rmse = float(np.sqrt(mean_squared_error(y, lgbm_dollar_preds)))
lgbm_mae = float(mean_absolute_error(y, lgbm_dollar_preds))
lgbm_r2 = float(r2_score(y, lgbm_dollar_preds))

cv_scores = cross_validate(best_lgbm_pipeline, X, y_log, cv=cv, return_train_score=True, scoring="r2")
train_r2 = float(cv_scores["train_score"].mean())
val_r2 = float(cv_scores["test_score"].mean())
overfit_gap = float(train_r2 - val_r2)

print("========================================")
print("🚀 MEMBER 4 (RAASHIDH) LIGHTGBM FINAL RESULTS")
print("========================================")
print(f"Baseline RMSE:        ${baseline_rmse:,.2f}")
print(f"LightGBM CV RMSE:     ${lgbm_rmse:,.2f}")
print(f"LightGBM CV MAE:      ${lgbm_mae:,.2f}")
print(f"LightGBM CV R²:       {lgbm_r2:.4f}")
print(f"Train R² Score:       {train_r2:.4f}")
print(f"Validation R² Score:  {val_r2:.4f}")
print(f"Overfit Gap:          {overfit_gap:.4f}")
print("========================================")

### CELL 7: Secondary Lens — Top 15 Value Drivers & Residuals

In [ ]:
# ========================================================
# CELL 7: SECONDARY LENS — TOP 15 VALUE DRIVERS & RESIDUALS
# ========================================================
best_lgbm_pipeline.fit(X, y_log)

encoder_step = best_lgbm_pipeline.named_steps["preprocessor"].named_steps["encode"]
feature_names = encoder_step.get_feature_names_out()
importances = best_lgbm_pipeline.named_steps["regressor"].feature_importances_

feature_imp_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False).head(15)

feature_imp_df["Clean_Name"] = feature_imp_df["Feature"].str.replace("numeric__", "").str.replace("nominal__", "").str.replace("indicators__", "")

Path("reports/figures").mkdir(parents=True, exist_ok=True)
Path("reports/Member_04_Raashidh").mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(12, 6))
sns.barplot(data=feature_imp_df, x="Importance", y="Clean_Name", palette="mako")
plt.title("Ames Housing: Top 15 Features Driving Property Prices (LightGBM)", fontsize=14, weight="bold")
plt.xlabel("Importance Score (Number of splits)")
plt.ylabel("House Feature")
plt.tight_layout()
plt.savefig("reports/figures/member4_top_features.png", dpi=300)
plt.savefig("reports/Member_04_Raashidh/member4_top_features.png", dpi=300)
plt.show()

residuals = y - lgbm_dollar_preds
plt.figure(figsize=(10, 5))
plt.scatter(lgbm_dollar_preds, residuals, alpha=0.4, color="teal")
plt.axhline(0, color="crimson", linestyle="--", linewidth=1.5)
plt.title("Residual Error Analysis — LightGBM (Predicted vs Error)", fontsize=13, weight="bold")
plt.xlabel("Predicted Sale Price ($)")
plt.ylabel("Prediction Error ($)")
plt.tight_layout()
plt.savefig("reports/figures/member4_residuals.png", dpi=300)
plt.savefig("reports/Member_04_Raashidh/member4_residuals.png", dpi=300)
plt.show()

### CELL 8: Save Artifacts for FastAPI, Report & Kaggle

In [ ]:
# ========================================================
# CELL 8: SAVE ARTIFACTS FOR FASTAPI, REPORT & KAGGLE
# ========================================================
Path("reports/model_results").mkdir(parents=True, exist_ok=True)
Path("notebooks/Member_04_Raashidh").mkdir(parents=True, exist_ok=True)

joblib.dump(best_lgbm_pipeline, "notebooks/Member_04_Raashidh/lightgbm_model.joblib")
joblib.dump(best_lgbm_pipeline, "lightgbm_model.joblib")
print("✅ Saved: notebooks/Member_04_Raashidh/lightgbm_model.joblib")

metrics_record = {
    "member": "Member 4 - Raashidh M.R.A. (IT24104169)",
    "model_name": "LightGBM Regressor",
    "baseline_benchmark": {
        "rmse": baseline_rmse,
        "mae": baseline_mae,
        "r2": baseline_r2
    },
    "lgbm_performance": {
        "rmse": lgbm_rmse,
        "mae": lgbm_mae,
        "r2": lgbm_r2,
        "train_r2": train_r2,
        "overfit_gap": overfit_gap
    },
    "best_parameters": grid_search.best_params_,
    "top_5_features": feature_imp_df["Clean_Name"].head(5).tolist()
}

with open("reports/model_results/member4_lgbm_metrics.json", "w") as f:
    json.dump(metrics_record, f, indent=4)

with open("reports/Member_04_Raashidh/member4_lgbm_metrics.json", "w") as f:
    json.dump(metrics_record, f, indent=4)

X_test = test_df.drop(columns=["Id"]).copy()
test_log_predictions = best_lgbm_pipeline.predict(X_test)
test_dollar_predictions = np.expm1(test_log_predictions)

submission_df = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": test_dollar_predictions
})
submission_df.to_csv("reports/Member_04_Raashidh/submission_lgbm.csv", index=False)
submission_df.to_csv("submission_lgbm.csv", index=False)
print("✅ Saved: submission_lgbm.csv")
print(submission_df.head())